# Evaluación de variables con valores faltantes

Este notebook compara el baseline de regresión logística del pipeline con versiones que retiran variables individualmente y por grupos. Usa las mismas cuatro particiones temporales y el mismo preprocesamiento que `07_entrenamiento.ipynb`. La métrica principal es ROC-AUC; PR-AUC y KS son complementarias.

El periodo de prueba (2017 en adelante) queda fuera de la selección. No se modifica el pipeline principal ni se eliminan columnas de sus archivos.

> **Actualización del criterio:** el flujo final ahora conserva 59 predictoras con menos del 10% de faltantes en entrenamiento + validación. Los grupos de variables de alta ausencia que se estudiaban aquí ya no pertenecen al conjunto actual; los resultados previos de ablación/calibración no deben tomarse como evidencia del nuevo baseline. La búsqueda de modelos debe volver a ejecutarse con la nueva lista.

## 1. Cargar el conjunto de desarrollo y reproducir el baseline

El archivo fuente contiene préstamos de varios años. Para estos experimentos se conserva únicamente el conjunto de desarrollo: préstamos elegibles otorgados antes de 2017, que abarca entrenamiento y validación. No se calculan métricas con los años reservados para prueba.

In [ ]:
from pathlib import Path
import sys
import gc
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.config import SEMILLA, ANIO_CORTE_PRUEBA, DIR_SALIDAS
from src.modelado import (
    NOMBRES_VARIABLES,
    cargar_datos_modelado,
    crear_tablas_imputadas_corte,
    construir_codificador,
    modelos_iniciales,
)
from src.evaluate import estadistico_ks

archivo = RAIZ / "accepted_2007_to_2018Q4.csv"
if not archivo.exists():
    from src.config import localizar_datos
    archivo = localizar_datos()

datos_completos, predictoras_baseline = cargar_datos_modelado(archivo)
datos = datos_completos.loc[datos_completos["anio"] < ANIO_CORTE_PRUEBA].copy()
del datos_completos
gc.collect()

assert datos["anio"].max() < ANIO_CORTE_PRUEBA
assert predictoras_baseline
print(f"Préstamos de desarrollo (entrenamiento + validación): {len(datos):,}")
print(f"Años incluidos: {datos['anio'].min()}–{datos['anio'].max()}")
print(f"Predictoras actuales del baseline: {len(predictoras_baseline)}")
print("Años de prueba excluidos de los experimentos: desde", ANIO_CORTE_PRUEBA)

modelo_baseline = modelos_iniciales()["regresion_logistica"]
print("Modelo de comparación:", modelo_baseline)

## 2. Variables candidatas y presencia en el baseline

El flujo actualizado conserva 59 predictoras con menos del 10% de faltantes en los préstamos elegibles de entrenamiento + validación. Las variables de los grupos A–D que superan el límite ya no forman parte del baseline actual, así que no se puede medir aquí el efecto de quitarlas una segunda vez. La tabla siguiente identifica cuáles están presentes y cuáles se excluyeron por el criterio actualizado.

In [ ]:
grupos = {
    "A": ["dti_joint", "annual_inc_joint", "verification_status_joint"],
    "B": ["mths_since_last_record", "mths_since_recent_bc_dlq", "il_util", "mths_since_last_major_derog"],
    "C": ["mths_since_rcnt_il", "all_util", "total_cu_tl", "open_acc_6m", "inq_last_12m",
          "open_rv_12m", "total_bal_il", "open_il_12m", "open_il_24m", "inq_fi", "max_bal_bc",
          "open_act_il", "open_rv_24m"],
    "D": ["mths_since_recent_revol_delinq", "mths_since_last_delinq", "mths_since_recent_inq"],
}

filas_estado = []
for grupo, variables in grupos.items():
    for variable in variables:
        filas_estado.append({
            "grupo": grupo,
            "variable_original": variable,
            "nombre_espanol": NOMBRES_VARIABLES.get(variable, variable),
            "incluida_en_baseline": variable in predictoras_baseline,
            "motivo_si_esta_excluida": (
                "No se conserva con el criterio actual (<10% de faltantes)"
                if variable not in predictoras_baseline else ""
            ),
        })
estado_variables = pd.DataFrame(filas_estado)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(estado_variables)

variables_activas = {g: [v for v in vs if v in predictoras_baseline] for g, vs in grupos.items()}
print("Variables del grupo B que sí están en el baseline:", variables_activas["B"])
print("Variables del grupo D que sí están en el baseline:", variables_activas["D"])
print("A y C no tienen variables dentro del baseline actual; sus efectos no se pueden medir por eliminación.")

## 3. Comprobar el patrón de faltantes por solicitud y por año

Esta auditoría descriptiva usa solo préstamos elegibles anteriores a 2017. En el grupo A se compara la disponibilidad con `application_type`. En el grupo C se resume el porcentaje de faltantes por año de otorgamiento. Estos resultados describen disponibilidad; no se presentan como evidencia de aporte predictivo.

In [ ]:
variables_auditoria = list(dict.fromkeys(sum(grupos.values(), [])))
columnas_auditoria = ["loan_status", "issue_d", "application_type"] + variables_auditoria
dtypes_auditoria = {c: "float64" for c in variables_auditoria}
dtypes_auditoria.update({
    "loan_status": "string",
    "issue_d": "string",
    "application_type": "string",
    "verification_status_joint": "string",
})
auditoria = pd.read_csv(
    archivo,
    usecols=columnas_auditoria,
    dtype=dtypes_auditoria,
    skipinitialspace=True,
    low_memory=False,
)
auditoria["issue_d"] = pd.to_datetime(auditoria["issue_d"], format="%b-%Y", errors="coerce")
mascara_desarrollo_auditoria = (
    auditoria["loan_status"].isin(["Fully Paid", "Charged Off"])
    & auditoria["issue_d"].dt.year.lt(ANIO_CORTE_PRUEBA)
)
auditoria = auditoria.loc[mascara_desarrollo_auditoria].copy()
auditoria["anio"] = auditoria["issue_d"].dt.year.astype("int16")
assert len(auditoria) == len(datos), "La auditoría y el pipeline no tienen las mismas filas de desarrollo."

resumen_faltantes = pd.DataFrame({
    "variable_original": variables_auditoria,
    "nombre_espanol": [NOMBRES_VARIABLES.get(v, v) for v in variables_auditoria],
    "registros_con_dato": [int(auditoria[v].notna().sum()) for v in variables_auditoria],
    "registros_faltantes": [int(auditoria[v].isna().sum()) for v in variables_auditoria],
})
resumen_faltantes["porcentaje_faltante"] = (
    resumen_faltantes["registros_faltantes"] / len(auditoria) * 100
)
print("Faltantes observados en entrenamiento + validación:")
display(resumen_faltantes.sort_values("porcentaje_faltante", ascending=False).round(4))

# Grupo A: disponibilidad por tipo de solicitud.
filas_aplicacion = []
for tipo, parte in auditoria.groupby("application_type", dropna=False, observed=True):
    for variable in grupos["A"]:
        filas_aplicacion.append({
            "tipo_solicitud": tipo,
            "variable": NOMBRES_VARIABLES.get(variable, variable),
            "solicitudes": len(parte),
            "con_dato": int(parte[variable].notna().sum()),
            "faltante_pct": float(parte[variable].isna().mean() * 100),
        })
tabla_aplicacion = pd.DataFrame(filas_aplicacion)
print("Disponibilidad del grupo A según tipo de solicitud:")
display(tabla_aplicacion.round(3))

# Grupo C: porcentaje faltante por año, limitado a 2007–2016.
por_anio = auditoria.groupby("anio", observed=True)
tabla_anual_C = pd.DataFrame(index=sorted(auditoria["anio"].unique()))
tabla_anual_C.index.name = "anio"
for variable in grupos["C"]:
    tabla_anual_C[NOMBRES_VARIABLES.get(variable, variable)] = por_anio[variable].apply(
        lambda s: s.isna().mean() * 100
    )
print("Porcentaje faltante del grupo C por año (%):")
display(tabla_anual_C.round(2))

del auditoria, mascara_desarrollo_auditoria, por_anio
gc.collect()

## 4. Diseño de la comparación

Se reproduce el baseline de regresión logística (`C=1`, `max_iter=2000`) con las cuatro particiones del notebook de entrenamiento: entrenar hasta 2012 y validar en 2013; hasta 2013 y validar en 2014; hasta 2014 y validar en 2015; hasta 2015 y validar en 2016. Para cada corte, medianas, indicadores, escalado y codificación se ajustan usando únicamente entrenamiento.

Cada eliminación retira también el indicador de faltante correspondiente (`variable__faltante`). Se mantienen las mismas filas de entrenamiento y validación. No se utiliza 2017 en adelante. Las diferencias se calculan como **métrica del experimento menos métrica baseline**; un valor negativo de ΔROC-AUC indica que quitar variables empeoró el ranking.

In [ ]:
# Experimentos que sí modifican el conjunto actual de predictoras.
experimentos_eliminacion = {
    "Sin grupo B": variables_activas["B"],
    "Sin grupo D": variables_activas["D"],
    "Sin grupos A, B y C": list(dict.fromkeys(
        variables_activas["A"] + variables_activas["B"] + variables_activas["C"]
    )),
}
for variable in variables_activas["B"] + variables_activas["D"]:
    experimentos_eliminacion[f"Sin {NOMBRES_VARIABLES.get(variable, variable)}"] = [variable]

# “Sin A, B y C” y “Sin B” son el mismo experimento aquí, porque A y C ya estaban fuera.
experimentos_unicos = {"Baseline": []}
for nombre, variables in experimentos_eliminacion.items():
    if variables and not any(set(variables) == set(v) for v in experimentos_unicos.values()):
        experimentos_unicos[nombre] = variables

resultados_detalle = []
puntuaciones = {nombre: [] for nombre in experimentos_unicos}
objetivos_validacion = []

for anio_validacion in range(2013, 2017):
    anio_final_entrenamiento = anio_validacion - 1
    print(f"Corte: entrenamiento hasta {anio_final_entrenamiento}; validación {anio_validacion}.", flush=True)
    tablas = crear_tablas_imputadas_corte(
        datos,
        predictoras_baseline,
        anio_final_entrenamiento=anio_final_entrenamiento,
        anio_validacion=anio_validacion,
    )
    y_train = tablas["y_entrenamiento"].to_numpy()
    y_valid = tablas["y_validacion"].to_numpy()
    objetivos_validacion.append(y_valid.copy())

    for nombre, eliminar in experimentos_unicos.items():
        columnas_fuera = list(eliminar) + [f"{v}__faltante" for v in eliminar]
        X_train = tablas["X_entrenamiento"].drop(columns=columnas_fuera, errors="ignore")
        X_valid = tablas["X_validacion"].drop(columns=columnas_fuera, errors="ignore")
        codificador = construir_codificador(X_train)
        X_train_codificado = codificador.fit_transform(X_train)
        X_valid_codificado = codificador.transform(X_valid)

        modelo = clone(modelo_baseline)
        inicio = time.perf_counter()
        modelo.fit(X_train_codificado, y_train)
        segundos = time.perf_counter() - inicio
        score = modelo.predict_proba(X_valid_codificado)[:, 1]
        puntuaciones[nombre].append(score.copy())
        resultados_detalle.append({
            "experimento": nombre,
            "anio_validacion": anio_validacion,
            "n_validacion": len(y_valid),
            "columnas_antes_de_codificar": X_train.shape[1],
            "segundos_ajuste": segundos,
            "roc_auc": roc_auc_score(y_valid, score),
            "pr_auc": average_precision_score(y_valid, score),
            "ks": estadistico_ks(y_valid, score),
        })
        del X_train, X_valid, X_train_codificado, X_valid_codificado, codificador, modelo, score
        gc.collect()
    del tablas, y_train, y_valid
    gc.collect()

resultados_detalle = pd.DataFrame(resultados_detalle)
print("Experimentos y cortes completados:", len(resultados_detalle))
display(resultados_detalle.round(4))

## 5. Resumen de métricas y diferencias frente al baseline

Esta sección conserva el diseño de comparación anterior, pero las variables estudiadas en los grupos A–D quedan excluidas por el nuevo límite de faltantes. Por tanto, no hay ablaciones aplicables ni diferencias de rendimiento que reportar para esas variables bajo la lista actual.

In [ ]:
resumen_metricas = (resultados_detalle.groupby("experimento")
    .agg(
        ROC_AUC=("roc_auc", "mean"),
        ROC_AUC_desviacion=("roc_auc", "std"),
        PR_AUC=("pr_auc", "mean"),
        KS=("ks", "mean"),
        cortes=("anio_validacion", "nunique"),
    )
)
base = resumen_metricas.loc["Baseline"]
resumen_metricas["delta_ROC_AUC"] = resumen_metricas["ROC_AUC"] - base["ROC_AUC"]
resumen_metricas["delta_PR_AUC"] = resumen_metricas["PR_AUC"] - base["PR_AUC"]
resumen_metricas["delta_KS"] = resumen_metricas["KS"] - base["KS"]

orden = ["Baseline", "Sin grupo B", "Sin grupo D", "Sin grupos A, B y C"]
orden += [f"Sin {NOMBRES_VARIABLES.get(v, v)}" for v in variables_activas["B"] + variables_activas["D"]]
filas_tabla = []
for nombre in orden:
    clave = nombre
    if nombre == "Sin grupos A, B y C":
        clave = "Sin grupo B"  # A y C ya estaban excluidos del baseline.
    if clave in resumen_metricas.index:
        fila = resumen_metricas.loc[clave].to_dict()
        if nombre == "Sin grupo B":
            estado = "Medido: se retiraron 2 activas; mths_since_last_record e il_util ya estaban excluidas"
        elif nombre == "Sin grupos A, B y C":
            estado = "Mismo ajuste que sin B: A y C ya estaban excluidos; se retiraron 2 variables activas de B"
        else:
            estado = "Medido"
    else:
        fila = {c: np.nan for c in resumen_metricas.columns}
        estado = "No evaluable: no había variables de este grupo en el baseline"
    filas_tabla.append({"experimento": nombre, **fila, "estado": estado})

# A y C no tienen eliminación evaluable; se incluyen para documentar el límite del análisis.
for nombre in ["Sin grupo A", "Sin grupo C"]:
    filas_tabla.append({
        "experimento": nombre,
        **{c: np.nan for c in resumen_metricas.columns},
        "estado": "No evaluable: las variables ya estaban excluidas del baseline",
    })
tabla_grupos = pd.DataFrame(filas_tabla)
columnas_mostrar = ["experimento", "ROC_AUC", "PR_AUC", "KS", "delta_ROC_AUC", "estado"]
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(tabla_grupos[columnas_mostrar].round({"ROC_AUC": 4, "PR_AUC": 4, "KS": 4, "delta_ROC_AUC": 6}))

print("Resultados individuales por corte:")
display(resultados_detalle.sort_values(["experimento", "anio_validacion"]).round(4))

# Comprobación opcional contra la ejecución ya guardada del notebook de entrenamiento.
ruta_metricas_previas = DIR_SALIDAS / "entrenamiento" / "metricas_validacion_temporal.csv"
if ruta_metricas_previas.exists():
    metricas_previas = pd.read_csv(ruta_metricas_previas)
    previas_logistica = metricas_previas.loc[metricas_previas["modelo"] == "regresion_logistica"]
    print("Promedios de regresión logística guardados previamente en el pipeline:")
    display(previas_logistica[["anio_validacion", "roc_auc", "pr_auc", "ks"]].round(4))

## 6. Recomendación por variable

Las variables que no aparecen en el baseline no pueden evaluarse mediante su eliminación. La decisión actual de excluirlas viene del criterio de cobertura menor al 10%, no de una prueba de que carezcan de valor predictivo. Si se desea reconsiderar alguna, deberá evaluarse en un experimento separado con el conjunto actual, las mismas particiones y la prueba reservada.

In [ ]:
# Regla transparente y conservadora: se revisan los cuatro signos de ΔROC-AUC,
# las métricas complementarias y la magnitud frente a la variación entre años.
filas_recomendacion = []
metricas_por_experimento = resumen_metricas.to_dict("index")
for grupo, variables in grupos.items():
    for variable in variables:
        porcentaje = resumen_faltantes.loc[
            resumen_faltantes["variable_original"] == variable, "porcentaje_faltante"
        ]
        pct = float(porcentaje.iloc[0]) if len(porcentaje) else np.nan
        if variable not in predictoras_baseline:
            decision = "Evaluar posteriormente"
            impacto = "No evaluado: ya estaba excluida del baseline"
            justificacion = "No se puede atribuirle un cambio por eliminación; no participó en los modelos comparados."
        else:
            escenario = f"Sin {NOMBRES_VARIABLES.get(variable, variable)}"
            filas_var = resultados_detalle[resultados_detalle["experimento"] == escenario]
            deltas = []
            for anio in range(2013, 2017):
                auc_base = resultados_detalle.loc[
                    (resultados_detalle["experimento"] == "Baseline")
                    & (resultados_detalle["anio_validacion"] == anio), "roc_auc"
                ].iloc[0]
                auc_sin = filas_var.loc[filas_var["anio_validacion"] == anio, "roc_auc"].iloc[0]
                deltas.append(auc_sin - auc_base)
            impacto_medio = float(np.mean(deltas))
            impacto = f"ΔROC-AUC {impacto_medio:+.6f}; por año: " + ", ".join(f"{d:+.6f}" for d in deltas)
            # El signo de la mejora al retirar la variable se contrasta también con PR-AUC y KS.
            resumen_var = metricas_por_experimento[escenario]
            delta_pr = resumen_var["PR_AUC"] - base["PR_AUC"]
            delta_ks = resumen_var["KS"] - base["KS"]
            if all(d < 0 for d in deltas):
                decision = "Conservar"
                justificacion = "Retirarla reduce ROC-AUC en los cuatro cortes; revisar además ΔPR-AUC y ΔKS."
            elif all(d > 0 for d in deltas) and delta_pr >= 0 and delta_ks >= 0:
                decision = "Eliminar"
                justificacion = "Candidata a retirar del baseline: la ablación no reduce métricas en estos cuatro cortes; las diferencias son muy pequeñas y no prueban una mejora significativa."
            else:
                decision = "Evaluar posteriormente"
                justificacion = "El efecto cambia entre años o las métricas complementarias no apuntan en la misma dirección."
        filas_recomendacion.append({
            "grupo": grupo,
            "variable": NOMBRES_VARIABLES.get(variable, variable),
            "variable_original": variable,
            "porcentaje_faltante": pct,
            "impacto_ROC_AUC": impacto,
            "decision": decision,
            "justificacion": justificacion,
        })

tabla_recomendaciones = pd.DataFrame(filas_recomendacion)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 90):
    display(tabla_recomendaciones)

salida = DIR_SALIDAS / "seleccion_variables_faltantes"
salida.mkdir(parents=True, exist_ok=True)
resultados_detalle.to_csv(salida / "metricas_por_corte.csv", index=False, encoding="utf-8-sig")
tabla_grupos.to_csv(salida / "comparacion_por_grupo.csv", index=False, encoding="utf-8-sig")
tabla_recomendaciones.to_csv(salida / "recomendacion_por_variable.csv", index=False, encoding="utf-8-sig")
estado_variables.to_csv(salida / "presencia_en_baseline.csv", index=False, encoding="utf-8-sig")
resumen_faltantes.to_csv(salida / "faltantes_candidatas_desarrollo.csv", index=False, encoding="utf-8-sig")
tabla_anual_C.to_csv(salida / "faltantes_grupo_C_por_anio.csv", encoding="utf-8-sig")
print("Tablas guardadas en:", salida)

## 7. Lectura y límites

- El conjunto de desarrollo termina en 2016; los años desde 2017 siguen reservados para la evaluación final.
- Las variables de los grupos A–D superan el límite actual de faltantes y no participan en el baseline de 59 predictoras.
- Las tablas descriptivas de disponibilidad muestran cuándo y para qué tipo de solicitud aparecen los datos; no demuestran que una variable mejore las predicciones.
- Las métricas de eliminación y calibración guardadas antes del cambio corresponden al baseline anterior de 64 variables. No deben mezclarse con los resultados que se obtengan al volver a ejecutar el modelo actualizado.

### 8. Experimento anterior de calibración — no aplicable al conjunto actual

Las variables `mths_since_recent_bc_dlq` y `mths_since_last_major_derog` superan el nuevo límite de faltantes y ya no están en las 59 predictoras. Por eso, el experimento anterior de quitar una o ambas del baseline no se vuelve a ejecutar: el resultado guardado antes de cambiar el criterio corresponde a otro conjunto de variables.

In [ ]:
print("Experimento de calibración omitido: las dos variables estudiadas quedaron excluidas por el umbral actual de faltantes.")

#### Resultados

No hay métricas nuevas para esta ablación. El resultado anterior corresponde al baseline de 64 variables y no debe compararse con el flujo actual.

In [ ]:
print("Sin ablación aplicable bajo la lista actual de predictoras.")

#### Curvas de calibración

No se generan curvas para variables que no están incluidas en el modelo actual. Para comparar probabilidades se requiere primero ejecutar el baseline actualizado en el notebook de entrenamiento.

In [ ]:
print("Curvas pendientes de la nueva ejecución del baseline.")

### 8.3. Conclusión del experimento

La recomendación de mantener o quitar estas dos variables se basaba en una regresión logística con la lista anterior de predictoras. Con el criterio nuevo, ambas quedan fuera por su porcentaje de faltantes, de modo que esas métricas anteriores no justifican una decisión sobre el baseline actualizado. Si más adelante se desea probarlas, debe hacerse como un experimento separado y comparar contra el mismo modelo, las mismas observaciones y los cortes de validación actuales. No se modifica ningún modelo desde este notebook.